# 4. Live Weather Tool

Replace mock weather data with a live Open-Meteo forecast.

In [ ]:
import json
import os
import time
from datetime import date, timedelta

from openai import OpenAI
from pydantic import BaseModel

from learning_langgraph.weather import get_weather

## Tool input

Live forecasts need exact dates. Open-Meteo supports today plus the next 15 days. The Python function returns a structured error when the dates are not supported.

In [ ]:
class WeatherInput(BaseModel):
    location: str
    start_date: date
    end_date: date

In [ ]:
api_key = os.getenv("AWS_BEARER_TOKEN_BEDROCK")

if not api_key:
    raise RuntimeError("AWS_BEARER_TOKEN_BEDROCK is required")

region = os.getenv("BEDROCK_REGION", "ap-south-1")
model_id = "google.gemma-3-4b-it"

client = OpenAI(
    api_key=api_key,
    base_url=f"https://bedrock-mantle.{region}.api.aws/v1",
)

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get a live weather forecast for a location and exact travel dates",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Destination city and country",
                },
                "start_date": {
                    "type": "string",
                    "format": "date",
                    "description": "First travel date in YYYY-MM-DD format",
                },
                "end_date": {
                    "type": "string",
                    "format": "date",
                    "description": "Last travel date in YYYY-MM-DD format",
                },
            },
            "required": ["location", "start_date", "end_date"],
            "additionalProperties": False,
        },
    },
}

session_usage = {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "latency_ms": 0,
}

In [ ]:
def call_model(messages: list, tools: list | None = None, tool_choice=None):
    start = time.perf_counter()
    request = {
        "model": model_id,
        "messages": messages,
        "temperature": 0.2,
        "max_tokens": 500,
    }

    if tools:
        request["tools"] = tools
        request["tool_choice"] = tool_choice or "auto"

    response = client.chat.completions.create(**request)
    latency_ms = round((time.perf_counter() - start) * 1000)
    usage = response.usage

    session_usage["calls"] += 1
    session_usage["input_tokens"] += usage.prompt_tokens
    session_usage["output_tokens"] += usage.completion_tokens
    session_usage["total_tokens"] += usage.total_tokens
    session_usage["latency_ms"] += latency_ms

    print("Finish reason:", response.choices[0].finish_reason)
    print("Call usage:", {
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_ms": latency_ms,
    })
    print("Session usage:", session_usage)
    return response

## Run the tool loop

The first model call must choose a tool. Python executes only allowed tools. The second model call receives the exact tool result and writes the answer.

In [ ]:
trip_start = date.today() + timedelta(days=3)
trip_end = trip_start + timedelta(days=3)

messages = [
    {
        "role": "system",
        "content": (
            "You are a personal travel decision assistant. "
            "Always use the weather tool for weather facts. "
            "Do not invent weather data. Mention the data source."
        ),
    },
    {
        "role": "user",
        "content": (
            f"I may visit Denpasar, Bali, Indonesia from {trip_start} to {trip_end} "
            "for beaches and outdoor activities. Is the weather suitable?"
        ),
    },
]

first_response = call_model(
    messages=messages,
    tools=[weather_tool],
    tool_choice="required",
)
assistant_message = first_response.choices[0].message
messages.append(assistant_message)

if not assistant_message.tool_calls:
    raise RuntimeError("The model did not request a tool")

for tool_call in assistant_message.tool_calls:
    if tool_call.function.name != "get_weather":
        raise ValueError(f"Tool is not allowed: {tool_call.function.name}")

    weather_input = WeatherInput.model_validate_json(
        tool_call.function.arguments
    )
    weather_result = get_weather(
        location=weather_input.location,
        start_date=weather_input.start_date,
        end_date=weather_input.end_date,
    )

    print("Tool input:")
    print(weather_input.model_dump_json(indent=2))
    print("Tool result:")
    print(json.dumps(weather_result, indent=2))

    messages.append(
        {
            "role": "user",
            "content": (
                "The get_weather tool returned this JSON:\n"
                f"{json.dumps(weather_result)}\n"
                "Use only this data to answer the original question."
            ),
        }
    )

final_response = call_model(messages=messages)
print("Final answer:")
print(final_response.choices[0].message.content)

## What changed

The LLM does not fetch weather itself. It only requests `get_weather`. Python owns validation, the external API call, and the returned JSON. This boundary makes the tool testable and reduces invented facts.

Gemma 3 on this Bedrock endpoint rejects a standard `tool` role in the follow-up conversation. This notebook sends the tool JSON as a `user` message, matching the workaround from stage 3.